In [1]:
import pandas as pd

postings = pd.read_csv('data/raw/job_postings.csv')
skills = pd.read_csv('data/raw/job_skills.csv')
summary = pd.read_csv('data/raw/job_summary.csv')

print("=== job_postings ===")
print(postings.shape)
print(postings.columns.tolist())
print(postings.head(2))

print("\n=== job_skills ===")
print(skills.shape)
print(skills.columns.tolist())
print(skills.head(2))

print("\n=== job_summary ===")
print(summary.shape)
print(summary.columns.tolist())
print(summary.head(2))

=== job_postings ===
(12217, 15)
['job_link', 'last_processed_time', 'last_status', 'got_summary', 'got_ner', 'is_being_worked', 'job_title', 'company', 'job_location', 'first_seen', 'search_city', 'search_country', 'search_position', 'job_level', 'job_type']
                                            job_link  \
0  https://www.linkedin.com/jobs/view/senior-mach...   
1  https://www.linkedin.com/jobs/view/principal-s...   

             last_processed_time   last_status got_summary got_ner  \
0  2024-01-21 08:08:48.031964+00  Finished NER           t       t   
1  2024-01-20 04:02:12.331406+00  Finished NER           t       t   

  is_being_worked                                     job_title  \
0               f              Senior Machine Learning Engineer   
1               f  Principal Software Engineer, ML Accelerators   

             company       job_location  first_seen search_city  \
0  Jobs for Humanity      New Haven, CT  2024-01-14  East Haven   
1             Aurora  Sa

In [2]:
df = postings.merge(skills, on='job_link', how='left') \
    .merge(summary, on='job_link', how='left')

print(df.shape)
print(df.columns.tolist())
df.head(2)

(12217, 17)
['job_link', 'last_processed_time', 'last_status', 'got_summary', 'got_ner', 'is_being_worked', 'job_title', 'company', 'job_location', 'first_seen', 'search_city', 'search_country', 'search_position', 'job_level', 'job_type', 'job_skills', 'job_summary']


,job_link,last_processed_time,last_status,got_summary,got_ner,is_being_worked,job_title,company,job_location,first_seen,search_city,search_country,search_position,job_level,job_type,job_skills,job_summary
0,https://www.linkedin.com/jobs/view/senior-mach...,2024-01-21 08:08:48.031964+00,Finished NER,t,t,f,Senior Machine Learning Engineer,Jobs for Humanity,"New Haven, CT",2024-01-14,East Haven,United States,Agricultural-Research Engineer,Mid senior,Onsite,"Machine Learning, Programming, Python, Scala, ...",Company Description\nJobs for Humanity is part...
1,https://www.linkedin.com/jobs/view/principal-s...,2024-01-20 04:02:12.331406+00,Finished NER,t,t,f,"Principal Software Engineer, ML Accelerators",Aurora,"San Francisco, CA",2024-01-14,El Cerrito,United States,Set-Key Driver,Mid senior,Onsite,"C++, Python, PyTorch, TensorFlow, MXNet, CUDA,...",Who We Are\nAurora (Nasdaq: AUR) is delivering...


In [3]:
# Check missing values per column
print(df.isnull().sum())

# Check distribution of job titles
print(df['job_title'].value_counts().head(20))

# Preview raw job summary text
print(df.loc[0, 'job_summary'])

job_link               0
last_processed_time    0
last_status            0
got_summary            0
got_ner                0
is_being_worked        0
job_title              0
company                0
job_location           1
first_seen             0
search_city            0
search_country         0
search_position        0
job_level              0
job_type               0
job_skills             5
job_summary            0
dtype: int64
job_title
Senior Data Engineer                                        285
Senior Data Analyst                                         163
Data Engineer                                               149
Senior MLOps Engineer                                       138
Data Analyst                                                137
Data Scientist                                              128
Lead Data Engineer                                          123
Senior Data Scientist                                       119
Data Architect                          

In [4]:
# Drop rows with missing critical fields
df = df.dropna(subset=['job_location', 'job_skills']).reset_index(drop=True)

# Check for duplicate postings
print(f"Duplicate job_links: {df['job_link'].duplicated().sum()}")
df = df.drop_duplicates(subset=['job_link']).reset_index(drop=True)

print(df.shape)

Duplicate job_links: 0
(12211, 17)


In [5]:
import re

def clean_job_summary(text):
    if not isinstance(text, str):
        return ""

    # Only strip boilerplate if it appears in the last 15% of the text
    # (safer than blanket removal from first occurrence)
    boilerplate_markers = [
        "show more", "equal opportunity employer",
        "protected veteran status", "reasonable accommodations",
    ]

    cutoff_point = int(len(text) * 0.85)
    tail = text[cutoff_point:].lower()

    for marker in boilerplate_markers:
        idx = text.lower().find(marker, cutoff_point)
        if idx != -1:
            text = text[:idx]
            break

    text = re.sub(r"\s+", " ", text).strip()
    return text

df['job_summary_clean'] = df['job_summary'].apply(clean_job_summary)

print(df['job_summary_clean'].str.len().describe())

count    12211.000000
mean      4256.025469
std       2293.421298
min         21.000000
25%       2541.500000
50%       3972.000000
75%       5708.000000
max      19177.000000
Name: job_summary_clean, dtype: float64


In [6]:
# Find rows where cleaning wiped everything out
empty_after_clean = df[df['job_summary_clean'].str.len() == 0]
print(f"Number of rows now empty: {len(empty_after_clean)}")

if len(empty_after_clean) > 0:
    print(empty_after_clean[['job_title', 'company']].head())
    print("\n--- Original text of first empty case ---")
    print(empty_after_clean.iloc[0]['job_summary'])

Number of rows now empty: 0


In [7]:
# Flag postings that are mostly procedural/administrative
admin_keywords = ['CalCareer', 'Examination/Employment Application', 'Statement of Qualifications']
df['is_admin_posting'] = df['job_summary'].str.contains('|'.join(admin_keywords), case=False, na=False)

print(f"Admin/procedural postings detected: {df['is_admin_posting'].sum()}")

df_filtered = df[~df['is_admin_posting']].reset_index(drop=True)

# Remove reposted duplicates (same title + company, different job_link)
before = df_filtered.shape[0]
df_filtered = df_filtered.drop_duplicates(subset=['job_title', 'company']).reset_index(drop=True)
after = df_filtered.shape[0]
print(f"Removed {before - after} duplicate postings (same title + company)")

print(df_filtered.shape)

Admin/procedural postings detected: 24
Removed 3363 duplicate postings (same title + company)
(8824, 19)


In [8]:
# Load your CV as reference text
with open('data/my_cv.txt', 'r', encoding='utf-8') as f:
    my_cv = f.read()

print(f"CV length: {len(my_cv)} characters")
print(my_cv[:300])  # quick preview

CV length: 4779 characters
Pharel Harold Nanseu Kombou

Data Science & Machine Learning – Working Student / Intern

Giessen, Germany | 01625971365 | haroldpharel@gmail.com | linkedin.com/in/pharel-nanseu-042281356 | github.com/Pharel8

PROFILE

Computer science student (B.Sc., THM Gießen, 5th semester) with a focus on Data Sc


In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Combine CV with all job summaries into one corpus
corpus = [my_cv] + df_filtered['job_summary_clean'].tolist()

vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
tfidf_matrix = vectorizer.fit_transform(corpus)

# CV is the first row (index 0), compare it against all job postings (index 1 onward)
cv_vector = tfidf_matrix[0:1]
job_vectors = tfidf_matrix[1:]

similarity_scores = cosine_similarity(cv_vector, job_vectors).flatten()

df_filtered['tfidf_score'] = similarity_scores

# Show top 10 matches
top_matches = df_filtered.sort_values('tfidf_score', ascending=False).head(10)
print(top_matches[['job_title', 'company', 'tfidf_score']])

                                              job_title  \
5811                          Machine Learning Engineer   
421               Senior Machine Learning Engineer - AI   
5513                   Senior Machine Learning Engineer   
8569                              Senior Data Scientist   
1532                   Sr II Machine Learning Scientist   
3725                          Machine Learning Engineer   
6791                   Senior Machine Learning Engineer   
2036                         Machine Learning Scientist   
5113  Data-Intensive Python/SQL Developer with Machi...   
1505                      Machine Learning Engineer III   

                                                company  tfidf_score  
5811  HummingBirds Consulting  LLC - now doing Busin...     0.327197  
421                             Recruiting from Scratch     0.327005  
5513                                              Lirio     0.318071  
8569                                         Optimizely     0.3102